# LoRA Fine-Tune GPT-2 — Custom BPE Tokenizer
**TDL Project — Week 3, Experiment 2**

This notebook:
1. Mounts Google Drive and reads `cleaned_data.txt`
2. Clones the project repo from GitHub
3. Installs dependencies
4. Copies the previous `gpt2_lora_finetuned` model from Drive (for 3-way comparison)
5. Re-tokenizes the corpus with the custom 12k BPE tokenizer
6. Resizes GPT-2 embeddings 50,257 → 12,000 and wraps with LoRA (r=8)
7. Trains for 5 epochs on 50k samples (~20 min on T4)
8. Produces 3-way comparison: pretrained GPT-2 vs finetuned orig-tok vs finetuned custom-tok
9. Saves model + results back to Drive

**Before running:**
- Confirm `My Drive/TDL-Project/data/cleaned_data.txt` exists
- Confirm `My Drive/TDL-Project/outputs/gpt2_lora_finetuned/` exists (from previous run)
- Runtime → Change runtime type → **T4 GPU**
- Then: Runtime → Run all

In [ ]:
# ── 0. Verify GPU ─────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU found — change runtime to T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT   = '/content/drive/MyDrive/TDL-Project'
CLEANED_DATA = f'{DRIVE_ROOT}/data/cleaned_data.txt'

assert os.path.exists(CLEANED_DATA), (
    f"cleaned_data.txt not found at {CLEANED_DATA}\n"
    f"Upload it to: My Drive/TDL-Project/data/cleaned_data.txt"
)
lines = open(CLEANED_DATA).readlines()
print(f"cleaned_data.txt found — {len(lines):,} lines")

In [ ]:
# ── 2. Clone repo & install deps ──────────────────────────────────────────
REPO_URL = 'https://github.com/Deekshithrathod/TDL-Project.git'
REPO_DIR = '/content/TDL-Project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print('Working dir:', os.getcwd())

In [ ]:
# ── 3. Install dependencies ────────────────────────────────────────────────
!pip install -q transformers datasets accelerate peft tokenizers emoji nltk tabulate

import nltk
nltk.download('words', quiet=True)
print('Dependencies installed.')

In [ ]:
# ── 4. Copy cleaned_data.txt into repo ────────────────────────────────────
import shutil
os.makedirs('data/processed', exist_ok=True)
shutil.copy(CLEANED_DATA, 'data/processed/cleaned_data.txt')
print('Copied cleaned_data.txt → data/processed/cleaned_data.txt')

In [ ]:
# ── 5. Copy previous LoRA model from Drive (needed for 3-way comparison) ──
# This is the gpt2_lora_finetuned model from the previous Colab run.
# If it's missing, the script will still run but skip the orig-tok column.
PREV_MODEL_SRC = f'{DRIVE_ROOT}/outputs/gpt2_lora_finetuned'
PREV_MODEL_DST = 'models/gpt2_lora_finetuned'

if os.path.exists(PREV_MODEL_SRC):
    os.makedirs('models', exist_ok=True)
    if os.path.exists(PREV_MODEL_DST):
        shutil.rmtree(PREV_MODEL_DST)
    shutil.copytree(PREV_MODEL_SRC, PREV_MODEL_DST)
    print(f'Copied previous model → {PREV_MODEL_DST}')
    print('Files:', os.listdir(PREV_MODEL_DST))
else:
    print(f'WARNING: Previous model not found at {PREV_MODEL_SRC}')
    print('3-way comparison will skip the finetuned_orig_tok column.')

In [ ]:
# ── 6. Fine-tune GPT-2 with custom BPE tokenizer ──────────────────────────
# tokenizer_codemixed and tokenizer_romanized_only had identical fertility
# (1.633) in Week 2 analysis — using codemixed as it covers the full corpus.
#
# batch_size=64 + max_train_samples=50000 → ~3,125 steps × 5 epochs
# Expected time on T4: ~20 minutes
!python scripts/finetune_gpt2_custom_tok.py \
    --tokenizer_path    tokenizers/tokenizer_codemixed \
    --cleaned_data      data/processed/cleaned_data.txt \
    --output            models/gpt2_lora_custom_tok \
    --prev_model_dir    models/gpt2_lora_finetuned \
    --epochs            5 \
    --rank              8 \
    --batch_size        64 \
    --max_train_samples 50000

In [ ]:
# ── 7. Show results ───────────────────────────────────────────────────────
import pandas as pd
import os

curve_path = 'report/perplexity_curve_custom_tok.csv'
comp_path  = 'report/custom_tok_comparison.csv'

if not os.path.exists(curve_path):
    print(f"ERROR: {curve_path} not found — training may have failed or not finished.")
    print("Check the output of Cell 6 for errors before proceeding.")
else:
    print('=== Perplexity Curve (custom tokenizer) ===')
    curve = pd.read_csv(curve_path)
    print(curve.to_string(index=False))

if not os.path.exists(comp_path):
    print(f"ERROR: {comp_path} not found — evaluation may have failed.")
    print("Check the output of Cell 6 for errors before proceeding.")
else:
    print('\n=== Three-Way Comparison (Telugu sentences) ===')
    comp = pd.read_csv(comp_path)
    cols = ['sentence', 'pretrained_gpt2_ppl', 'finetuned_orig_tok_ppl', 'finetuned_custom_tok_ppl']
    print(comp[cols].to_string(index=False))

    print(f'\nAvg pretrained GPT-2 PPL:       {comp["pretrained_gpt2_ppl"].mean():.1f}')
    if comp['finetuned_orig_tok_ppl'].notna().any():
        print(f'Avg finetuned orig-tok PPL:     {comp["finetuned_orig_tok_ppl"].mean():.1f}')
    print(f'Avg finetuned custom-tok PPL:   {comp["finetuned_custom_tok_ppl"].mean():.1f}')
    print('\nNOTE: PPL values are not directly comparable across tokenizers.')

In [ ]:
# ── 8. Save everything back to Drive ──────────────────────────────────────
import os, shutil

OUT_DIR = f'{DRIVE_ROOT}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

# Model adapter weights
model_src = 'models/gpt2_lora_custom_tok'
model_dst = f'{OUT_DIR}/gpt2_lora_custom_tok'
if os.path.exists(model_src) and any(os.scandir(model_src)):
    if os.path.exists(model_dst):
        shutil.rmtree(model_dst)
    shutil.copytree(model_src, model_dst)
    print(f'Model saved to Drive: {model_dst}')
else:
    print(f'WARNING: {model_src} is empty or missing — skipping model save.')

# Report CSVs
for fname in ['perplexity_curve_custom_tok.csv', 'custom_tok_comparison.csv']:
    src = f'report/{fname}'
    dst = f'{OUT_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved: {dst}')
    else:
        print(f'WARNING: {src} not found — skipping.')

# trainer_state.json
ckpt_src = 'models/gpt2_lora_custom_tok_checkpoints'
if os.path.exists(ckpt_src):
    for root, dirs, files in os.walk(ckpt_src):
        if 'trainer_state.json' in files:
            shutil.copy(
                os.path.join(root, 'trainer_state.json'),
                f'{OUT_DIR}/trainer_state_custom_tok.json'
            )
            print(f'Saved: {OUT_DIR}/trainer_state_custom_tok.json')
            break

print('\nDone.')

## After training completes

Download from Drive (`My Drive/TDL-Project/outputs/`):
- `gpt2_lora_custom_tok/` → copy to `models/gpt2_lora_custom_tok/` locally
- `perplexity_curve_custom_tok.csv` → copy to `report/`
- `custom_tok_comparison.csv` → copy to `report/`

Then let Claude commit the results and update `History.md`.